# 07. Minimum Spanning Tree and Bipartite Graphs

**MST:** connect all vertices with the **cheapest total edge weight** and no cycles. **Bipartite:** can the vertices be split into two groups with every edge going between the groups?

**Learning order:** REAL LIFE → MST PROPERTIES → KRUSKAL → PRIM → CONNECT POINTS → BIPARTITE → POSSIBLE BIPARTITION → INTERVIEW THINKING

## 1. Real life and properties

- **Laying cables / roads / pipes** to connect all houses with the least cost.
- **Network design, clustering.**

A spanning tree of `V` vertices has exactly **`V - 1` edges**. Two key facts:
- **Cut property:** the cheapest edge crossing any split of the vertices is in some MST.
- **Cycle property:** the heaviest edge on a cycle is not needed.

Both Kruskal and Prim are **greedy** and correct because of these.

## 2. Kruskal's algorithm (sort edges + Union-Find)

1. Sort edges by weight.
2. For each edge, if its endpoints are in **different** components (`union` succeeds), take it.
3. Stop after `V - 1` edges.

O(E log E). Best for sparse graphs given as an edge list.

In [1]:
import heapq
from collections import defaultdict

class DSU:
    def __init__(self, n):
        self.parent, self.size, self.count = list(range(n)), [1] * n, n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.count -= 1
        return True

def kruskal(n, edges):                              # edges: (u, v, w)
    dsu, total, used = DSU(n), 0, 0
    for u, v, w in sorted(edges, key=lambda e: e[2]):
        if dsu.union(u, v):
            total += w
            used += 1
            if used == n - 1:
                break
    return total if used == n - 1 else -1           # -1: graph is not connected

edges = [(0, 1, 10), (0, 2, 6), (0, 3, 5), (1, 3, 15), (2, 3, 4)]
print(kruskal(4, edges))   # 19

19


## 3. Prim's algorithm (grow one tree with a heap)

Start from any vertex. Repeatedly add the **cheapest edge** that connects the tree to a **new** vertex (min-heap of edges leaving the tree). O(E log V). Best for dense graphs or adjacency lists. It looks like Dijkstra, but the heap key is the **edge weight**, not the distance from the source.

In [2]:
def prim(n, edges):
    graph = defaultdict(list)
    for u, v, w in edges:
        graph[u].append((v, w))
        graph[v].append((u, w))
    visited, heap, total = set(), [(0, 0)], 0       # (edge weight, vertex)
    while heap and len(visited) < n:
        w, u = heapq.heappop(heap)
        if u in visited:
            continue
        visited.add(u)
        total += w
        for v, wt in graph[u]:
            if v not in visited:
                heapq.heappush(heap, (wt, v))
    return total if len(visited) == n else -1

print(prim(4, edges))   # 19

19


## 4. Min Cost to Connect All Points

Points on a plane, edge cost = Manhattan distance. Every pair is an edge (a **dense** graph), so Prim in O(n²) without a heap is ideal. Keep `min_dist[v]` = cheapest edge connecting `v` to the tree.

In [3]:
def min_cost_connect_points(points):
    n = len(points)
    min_dist = [float("inf")] * n
    min_dist[0] = 0
    in_tree = [False] * n
    total = 0
    for _ in range(n):
        u = min((i for i in range(n) if not in_tree[i]), key=lambda i: min_dist[i])
        in_tree[u] = True
        total += min_dist[u]
        for v in range(n):
            if not in_tree[v]:
                d = abs(points[u][0] - points[v][0]) + abs(points[u][1] - points[v][1])
                min_dist[v] = min(min_dist[v], d)
    return total

print(min_cost_connect_points([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]))   # 20

20


## 5. Bipartite graph (two-coloring)

A graph is bipartite **iff it has no odd-length cycle**. Test by coloring: BFS from each unvisited vertex, give neighbours the **opposite color**. If a neighbour already has the **same** color, it is not bipartite.

Real life: can we split people into two teams so that **enemies** are never on the same team? Can a graph of tasks be split between two machines with no conflicts?

In [4]:
from collections import deque

def is_bipartite(graph):                            # graph: adjacency list (list of lists)
    color = {}
    for start in range(len(graph)):
        if start in color:
            continue
        color[start] = 0
        q = deque([start])
        while q:
            u = q.popleft()
            for v in graph[u]:
                if v not in color:
                    color[v] = 1 - color[u]
                    q.append(v)
                elif color[v] == color[u]:
                    return False
    return True

print(is_bipartite([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]]), is_bipartite([[1, 3], [0, 2], [1, 3], [0, 2]]))   # False True

False True


## 6. Possible Bipartition

`dislikes[i] = [a, b]`: `a` and `b` cannot be in the same group. This is exactly bipartite checking on the "dislike" graph.

In [5]:
def possible_bipartition(n, dislikes):
    graph = [[] for _ in range(n)]
    for a, b in dislikes:
        graph[a - 1].append(b - 1)
        graph[b - 1].append(a - 1)
    return is_bipartite(graph)

print(possible_bipartition(4, [[1, 2], [1, 3], [2, 4]]), possible_bipartition(3, [[1, 2], [1, 3], [2, 3]]))   # True False

True False


## 7. Interview thinking

**Signals (MST):** "connect all with minimum total cost", "cables/roads", "cheapest way to join".
**Signals (bipartite):** "split into two groups", "no conflicts within a group", "odd cycle".

**Say:** "This is a minimum spanning tree. I'll use Kruskal: sort edges and union components. O(E log E)." / "I'll 2-color the graph with BFS; a conflict means an odd cycle."

**Common mistakes:** applying MST to a disconnected graph without checking `used == n - 1`, confusing MST with shortest path (MST minimizes the **total**, not individual distances), forgetting disconnected components in the bipartite check, 1-indexed people (`a - 1`).

## Quick Revision

- MST has `V - 1` edges. Kruskal = sort + DSU. Prim = heap of crossing edges.
- Dense geometric graph → Prim O(n²).
- Bipartite ⇔ no odd cycle ⇔ 2-colorable by BFS.
- MST ≠ shortest paths.